# Delivery duration across key factors

This notebook compares `Delivery_Time` across traffic, weather, estimated straight-line distance, and other available categories. Every group summary includes the number of records and the mean, median, and 90th percentile. The notebook reads the approved CSV from the user's Downloads folder and does not modify it.

`Delivery_Time` is treated as delivery duration; its unit is unconfirmed and is left unspecified. `Pickup_Delay_Minutes` means order-to-pickup duration in minutes. The dataset has no promised delivery time, actual delivery timestamp, or delivery status, so this analysis does not calculate on-time, late, completion, or promise-based delay rates. Group differences are descriptive and do not establish causes.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

csv_path = Path.home() / "Downloads" / "amazon_delivery_cleaned_updated.csv"
repo_root = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
output_dir = repo_root / "outputs"
output_dir.mkdir(exist_ok=True)
df = pd.read_csv(csv_path)
assert df.shape == (43_739, 28), f"Unexpected approved-file shape: {df.shape}"
assert df["Delivery_Time"].notna().all(), "Delivery_Time has missing values; review before proceeding"
print(f"Source: {csv_path}")
print(f"Dataset: {len(df):,} records, {df.shape[1]} columns")
print("Delivery_Time unit: unspecified (not verified)")

## Group summaries

Counts include only records with both a non-missing grouping value and a non-missing `Delivery_Time`. Missing category values are excluded from category comparisons and are counted separately below. The 90th percentile is included to show the upper portion of each group’s observed duration distribution. `Pickup_Delay_Minutes` is included as an observed order-to-pickup duration grouping in minutes; it is not a promised-time delay measure.

In [ ]:
def duration_summary(data, group):
    usable = data.loc[data[group].notna(), [group, "Delivery_Time"]]
    result = usable.groupby(group, observed=True)["Delivery_Time"].agg(
        records="count", mean="mean", median="median", p90=lambda s: s.quantile(.90),
        minimum="min", maximum="max"
    ).sort_values("median")
    return result.round(2)

group_fields = ["Traffic", "Weather", "Vehicle", "Area", "Order_Time_Period", "Category", "Pickup_Delay_Minutes"]
for field in group_fields:
    print(f"\n{field}")
    display(duration_summary(df, field))

print("\nMissing group values (not included in category summaries)")
display(df[group_fields].isna().sum().rename("missing_records").to_frame())

## Estimated straight-line distance

There is no distance field in the approved CSV. This section calculates great-circle distance from store and drop coordinates for rows where all four coordinates are present and within geographic bounds. It is an estimate of straight-line distance, not road distance or route length. Distance bands use quartiles of the usable sample; ties can result in fewer than four bands.

In [ ]:
lat_cols = ["Store_Latitude", "Drop_Latitude"]
lon_cols = ["Store_Longitude", "Drop_Longitude"]
coords = df[lat_cols + lon_cols].apply(pd.to_numeric, errors="coerce")
valid = (
    coords["Store_Latitude"].between(-90, 90) & coords["Drop_Latitude"].between(-90, 90) &
    coords["Store_Longitude"].between(-180, 180) & coords["Drop_Longitude"].between(-180, 180)
)
rad = np.radians
lat1, lat2 = rad(coords["Store_Latitude"]), rad(coords["Drop_Latitude"])
dlat = lat2 - lat1
dlon = rad(coords["Drop_Longitude"] - coords["Store_Longitude"])
a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
distance_km = 6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
df["Estimated_Straight_Line_Distance_km"] = np.where(valid, distance_km, np.nan)
distance_sample = df.loc[df["Estimated_Straight_Line_Distance_km"].notna()].copy()
distance_sample["Distance_Band"] = pd.qcut(
    distance_sample["Estimated_Straight_Line_Distance_km"], q=4,
    labels=["Q1 (shorter)", "Q2", "Q3", "Q4 (longer)"], duplicates="drop"
)
print(f"Records with valid coordinate pairs: {len(distance_sample):,} of {len(df):,}")
print(f"Estimated distances above 50 km: {(distance_sample['Estimated_Straight_Line_Distance_km'] > 50).sum():,}")
print(f"Maximum estimated straight-line distance (km): {distance_sample['Estimated_Straight_Line_Distance_km'].max():,.2f}")
print("Estimated straight-line distance (km), usable records:")
display(distance_sample["Estimated_Straight_Line_Distance_km"].describe(percentiles=[.25,.5,.75,.9]).to_frame())
print("Delivery duration by estimated straight-line distance band:")
display(duration_summary(distance_sample, "Distance_Band"))

## Visual comparisons

The plots show group medians and interquartile ranges (25th–75th percentiles), with sample counts in the labels. Values are in the recorded `Delivery_Time` scale; its unit remains unspecified.

In [ ]:
def plot_groups(data, group, title, filename, sort_by="median"):
    usable = data.loc[data[group].notna(), [group, "Delivery_Time"]]
    order = usable.groupby(group, observed=True)["Delivery_Time"].median().sort_values().index
    stats = usable.groupby(group, observed=True)["Delivery_Time"].agg(
        n="count", median="median", q1=lambda s: s.quantile(.25), q3=lambda s: s.quantile(.75)
    ).reindex(order)
    fig, ax = plt.subplots(figsize=(9, max(4, .42 * len(stats))))
    y = np.arange(len(stats))
    ax.hlines(y, stats["q1"], stats["q3"], color="#86a9c2", linewidth=7, alpha=.85)
    ax.scatter(stats["median"], y, color="#163b5c", s=36, zorder=3)
    ax.set_yticks(y, [f"{label} (n={int(n):,})" for label,n in zip(stats.index, stats["n"])])
    ax.set_xlabel("Delivery_Time (unit unconfirmed)")
    ax.set_title(title)
    ax.grid(axis="x", alpha=.25)
    fig.tight_layout()
    fig.savefig(output_dir / filename, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plot_groups(df, "Traffic", "Delivery duration by traffic category", "delivery_time_by_traffic.png")
plot_groups(df, "Weather", "Delivery duration by observed weather category", "delivery_time_by_weather.png")
plot_groups(distance_sample, "Distance_Band", "Delivery duration by estimated straight-line distance band", "delivery_time_by_distance_band.png")
plot_groups(df, "Area", "Delivery duration by area category", "delivery_time_by_area.png")
plot_groups(df, "Order_Time_Period", "Delivery duration by order time period", "delivery_time_by_order_period.png")
plot_groups(df, "Vehicle", "Delivery duration by vehicle category", "delivery_time_by_vehicle.png")

## Interpretation limits

- These are descriptive comparisons from this file. They do not show that a category causes longer or shorter delivery duration.
- Category groups have different sample sizes. Use the counts alongside all summary statistics; interpret the very small vehicle and area groups cautiously.
- The 91 rows with missing `Traffic`, `Weather`, and `Order_Time_Period` are excluded only from summaries for those fields; no values are imputed.
- Distance is derived from available coordinates and represents great-circle straight-line distance. It is not road distance. Records without complete valid coordinate pairs are excluded from distance comparisons.
- `Delivery_Time` unit is unconfirmed. Do not add a time unit label without source documentation.
- The available data do not support on-time, late, completion, or promise-based delay rates.